In [1]:
pip install dash-bootstrap-components


  Using cached dash_bootstrap_components-2.0.2-py3-none-any.whl.metadata (18 kB)
Using cached dash_bootstrap_components-2.0.2-py3-none-any.whl (202 kB)
Note: you may need to restart the kernel to use updated packages.


In [45]:
# 📦 Core Imports
import dash
from dash import html, dcc, Input, Output, State, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import base64
import tempfile

# 🧠 Your scheduling logic
from scheduling import (
    fcfs, sjf, priority_scheduling, round_robin, priority_rr,
    generate_random_processes, read_processes_from_file, Process
)


In [46]:
# 🖥️ Initialize Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.SANDSTONE])
app.title = "CPU Scheduler Simulator"

# 📚 Scheduling algorithm map
algorithms = {
    "FCFS": fcfs,
    "SJF": sjf,
    "Priority": priority_scheduling,
    "Round Robin": round_robin,
    "Priority + RR": priority_rr
}


In [47]:
# 🧱 UI Layout
app.layout = dbc.Container([
    html.H1("CPU Scheduler Simulator", className="text-center text-primary my-4"),

    dbc.Row([
        # 🔧 Configuration Panel
        dbc.Col([
            html.H5("Configuration", className="text-secondary mb-3"),

            html.Label("Select Input Method"),
            dcc.Dropdown(
                options=[
                    {"label": "Upload .txt File", "value": "file"},
                    {"label": "Manual / Random Input", "value": "manual"}
                ],
                id="input-method",
                value="manual",
                className="mb-3"
            ),

            html.Div(id="file-upload-div", children=[
                dcc.Upload(
                    id='upload-data',
                    children=html.Div(['Drag and Drop or ', html.A('Select File')]),
                    style={
                        'width': '100%', 'height': '60px', 'lineHeight': '60px',
                        'borderWidth': '1px', 'borderStyle': 'dashed',
                        'borderRadius': '5px', 'textAlign': 'center'
                    },
                    multiple=False
                )
            ], style={'display': 'none'}, className="mb-3"),

            html.Div(id="manual-input-div", children=[
                html.Label("Number of Processes"),
                dcc.Input(id="num-processes", type="number", min=1, value=5, className="form-control mb-3"),

                html.Label("Generate values randomly?"),
                dcc.Checklist(
                    id="random-values",
                    options=[{"label": "Yes", "value": "yes"}],
                    value=["yes"],
                    className="mb-3"
                ),
                html.Div(id="manual-fields-div")
            ]),

            html.Label("Select Scheduling Algorithm"),
            dcc.Dropdown(
                options=[{"label": k, "value": k} for k in algorithms.keys()],
                value="FCFS",
                id="algorithm",
                className="mb-3"
            ),

            html.Label("Quantum (used for RR and Priority + RR)"),
            dcc.Input(
                id="quantum",
                type="number",
                min=0.01,
                step=0.01,
                value=1,
                className="form-control mb-3"
            ),

            dbc.Button("Run Simulation", id="run-button", color="success", className="w-100")
        ], md=4),

        # 📊 Results Panel
        dbc.Col([
            html.H5("Initial Process Table", className="text-secondary"),
            dash_table.DataTable(id="process-table", style_table={'overflowX': 'auto'}),
            html.Div(id="metrics-output", className="my-3"),
            dcc.Graph(id="waiting-bar-chart"),
            dcc.Graph(id="turnaround-bar-chart"),
            dcc.Graph(id="usage-chart"),
            dcc.Graph(id="gantt-chart")
        ], md=8)
    ]),

    html.Hr(),
    html.H3("Compare Algorithms", className="text-secondary"),
    dbc.Button("Compare All Algorithms", id="compare-btn", color="primary", className="mb-3"),
    dcc.Graph(id="comparison-chart"),
    dcc.Store(id='stored-processes')
], fluid=True)


In [48]:
# 🔍 Decode uploaded file content
def decode_contents(contents):
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)
    return decoded.decode("utf-8")


In [49]:
# 🎛️ Toggle file/manual input display
@app.callback(
    Output("file-upload-div", "style"),
    Output("manual-input-div", "style"),
    Input("input-method", "value")
)
def toggle_input_fields(input_method):
    return (
        {"display": "block"} if input_method == "file" else {"display": "none"},
        {"display": "block"} if input_method == "manual" else {"display": "none"}
    )


In [50]:
@app.callback(
    Output("manual-fields-div", "children"),
    Input("num-processes", "value"),
    Input("random-values", "value")
)
def generate_manual_fields(num, random):
    if not num or "yes" in random:
        return ""

    fields = []
    for i in range(num):
        fields.append(html.Div([
            html.H5(f"Process {i + 1}"),
            html.Label("Arrival Time"),
            dcc.Input(id={'type': 'arrival', 'index': i}, type="number", min=0, step=1),
            html.Label("Burst Time"),
            dcc.Input(id={'type': 'burst', 'index': i}, type="number", min=1, step=1),
            html.Label("Priority"),
            dcc.Input(id={'type': 'priority', 'index': i}, type="number", min=0, step=1, value=0),
        ], style={'marginBottom': '15px'}))
    return fields


In [51]:
# 🧮 Main Simulation Callback
@app.callback(
    Output("process-table", "data"),
    Output("process-table", "columns"),
    Output("metrics-output", "children"),
    Output("waiting-bar-chart", "figure"),
    Output("turnaround-bar-chart", "figure"),
    Output("usage-chart", "figure"),
    Output("gantt-chart", "figure"),
    Output("stored-processes", "data"),
    Input("run-button", "n_clicks"),
    State("input-method", "value"),
    State("upload-data", "contents"),
    State("num-processes", "value"),
    State("random-values", "value"),
    State("algorithm", "value"),
    State("quantum", "value"),
    State({'type': 'arrival', 'index': dash.ALL}, 'value'),
    State({'type': 'burst', 'index': dash.ALL}, 'value'),
    State({'type': 'priority', 'index': dash.ALL}, 'value')
)

def run_simulation(n_clicks, input_method, file_content, num, random_opt, algorithm, quantum, arrivals, bursts, priorities):
    if not n_clicks:
        return [], [], "", {}, {}, {}, {}, []

    try:
        if input_method == "file":
            if file_content is None:
                return [], [], "No file uploaded.", {}, {}, {}, {}, []
            content_str = decode_contents(file_content)
            with tempfile.NamedTemporaryFile(delete=False, suffix=".txt") as temp:
                temp.write(content_str.encode("utf-8"))
                temp.flush()
                processes = read_processes_from_file(temp.name)
        else:
            if "yes" in random_opt:
                processes = generate_random_processes(num)
            else:
                processes = [Process(i + 1, arrivals[i], bursts[i], priorities[i]) for i in range(num)]

        func = algorithms[algorithm]
        result = func(processes.copy()) if "RR" not in algorithm else func(processes.copy(), quantum)

        if isinstance(result, tuple) and len(result) == 5:
            schedule, avg_tat, avg_wt, cpu_util, timeline = result
        else:
            schedule, avg_tat, avg_wt, cpu_util = result
            timeline = [(p.pid, p.start_time, p.completion_time) for p in processes if p.start_time is not None and p.completion_time is not None]

        df = pd.DataFrame([{
            "PID": p.pid,
            "Arrival": p.arrival_time,
            "Burst": p.burst_time,
            "Priority": p.priority,
            "Remaining": p.remaining_time,
            "Completion": p.completion_time or '-',
            "Waiting": p.waiting_time,
            "Turnaround": p.turnaround_time,
            "Start": p.start_time or p.arrival_time
        } for p in processes])

        stored_data = [dict(
            pid=p.pid,
            arrival_time=p.arrival_time,
            burst_time=p.burst_time,
            priority=p.priority,
            remaining_time=p.remaining_time
        ) for p in processes]

        bar_waiting = px.bar(df, x="PID", y="Waiting", title="Waiting Time per Process")
        bar_turnaround = px.bar(df, x="PID", y="Turnaround", title="Turnaround Time per Process")

        # Gantt chart with consistent colors and labels
        gantt_data = []
        timeline = sorted(timeline, key=lambda x: x[1])  # sort by start time
        current_time = 0
        color_map = {}
        colors = px.colors.qualitative.Plotly
        color_idx = 0

        for pid, start, end in timeline:
            if start > current_time:
                gantt_data.append({
                    "Task": "CPU", "Start": current_time, "Finish": start,
                    "Label": "Idle", "Color": "gray"
                })
            pid_label = f"P{pid}"
            if pid_label not in color_map:
                color_map[pid_label] = colors[color_idx % len(colors)]
                color_idx += 1
            gantt_data.append({
                "Task": "CPU", "Start": start, "Finish": end,
                "Label": pid_label, "Color": color_map[pid_label]
            })
            current_time = max(current_time, end)

        gantt_fig = go.Figure()
        for segment in gantt_data:
            gantt_fig.add_trace(go.Bar(
                x=[segment["Finish"] - segment["Start"]],
                y=["CPU"],
                base=segment["Start"],
                orientation='h',
                name=segment["Label"],
                marker=dict(color=segment["Color"]),
                text=segment["Label"],
                textposition='inside',
                hovertemplate=f'{segment["Label"]}: [{segment["Start"]}, {segment["Finish"]}]<extra></extra>'
            ))


        gantt_fig.update_layout(
            title="Gantt Chart of CPU Usage",
            barmode='stack',
            xaxis_title="Time",
            showlegend=True,
            height=300
        )

        usage_fig = go.Figure()
        for i, (pid, start, end) in enumerate(timeline):
            usage_fig.add_trace(go.Scatter(
                x=[start, end],
                y=[pid, pid],
                mode='lines',
                line=dict(width=10, color=colors[pid % len(colors)]),
                name=f"P{pid}",
                showlegend=False
            ))

        usage_fig.update_layout(
            title="CPU Usage Over Time",
            xaxis_title="Time",
            yaxis_title="Process ID",
            yaxis=dict(tickmode='linear', dtick=1),
            height=400
        )

        summary = html.Div([
            html.P(f"Avg Turnaround Time: {avg_tat:.2f}"),
            html.P(f"Avg Waiting Time: {avg_wt:.2f}"),
            html.P(f"CPU Utilization: {cpu_util:.2f}%")
        ])

        columns = [{"name": i, "id": i} for i in df.columns]
        return df.to_dict("records"), columns, summary, bar_waiting, bar_turnaround, usage_fig, gantt_fig, stored_data

    except Exception as e:
        return [], [], f"Error: {str(e)}", {}, {}, {}, {}, []


In [52]:
# 📊 Comparison Callback
@app.callback(
    Output("comparison-chart", "figure"),
    Input("compare-btn", "n_clicks"),
    State("stored-processes", "data"),
    State("quantum", "value")
)

def compare_algorithms(n_clicks, stored_data, quantum):
    if not n_clicks or not stored_data:
        return go.Figure()

    try:
        processes = [
            Process(p["pid"], p["arrival_time"], p["burst_time"], p["priority"])
            for p in stored_data
        ]

        results = {}
        for algo in algorithms:
            func = algorithms[algo]
            result = func(processes.copy()) if "RR" not in algo else func(processes.copy(), quantum)
            avg_tat, avg_wt, cpu_util = result[1], result[2], result[3]
            results[algo] = {"TAT": avg_tat, "WT": avg_wt, "CPU": cpu_util}

        categories = ["TAT", "WT", "CPU"]
        fig = go.Figure()
        for algo in algorithms:
            fig.add_trace(go.Bar(
                x=categories,
                y=[results[algo]["TAT"], results[algo]["WT"], results[algo]["CPU"]],
                name=algo
            ))
        fig.update_layout(title="Algorithm Comparison", yaxis_title="Value", barmode='group')
        return fig

    except Exception as e:
        return go.Figure()


In [53]:
app.run(debug=True)